## Semantic search

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import glob
from pprint import pprint 
from langchain_community.document_loaders import PyPDFLoader

pdf_files = glob.glob(r"C:\\Users\\dell\\Documents\\Agentic-AI-Langchain\\Content\\*.pdf")
loader = PyPDFLoader(pdf_files[0])  # Load the first PDF file

data = loader.load()

pprint(data)

C:\Users\dell\AppData\Local\Temp\ipykernel_22400\4245289029.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T15:41:33+05:30', 'author': '', 'moddate': '2026-06-02T15:41:33+05:30', 'title': 'Microsoft Word - Concept of agent tools and memory in langchain', 'source': 'C:\\\\Users\\\\dell\\\\Documents\\\\Agentic-AI-Langchain\\\\Content\\Concept of agent tools and memory in langchain.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Concept of agent tools and memory in langchain \nAuthor – Sambita Chakraborty \nIn LangChain, Agents act as dynamic reasoning engines that use a Large Language Model (LLM) to decide \na sequence of acƟons. Unlike rigid chains that follow a hardcoded path, an agent evaluates user input, \ndecides which Tools to use, executes them, and passes the results back into its Memory to plan the next \nstep, \nRef: LangChain Deep Dive: Chains, Tools, Agents & Memory | by Amit Kharche | Medium \n        Workﬂows and agents - Docs by LangChain \n \n 1. The 

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

13


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [5]:
import os
from langchain_chroma import Chroma

CHROMA_DB_PATH = "./chroma_db"

# Always open or create the persistent DB
vector_store = Chroma(
    persist_directory=CHROMA_DB_PATH,
    embedding_function=embeddings
)

# Find which source files are already indexed
existing = vector_store.get()
indexed_sources = {m.get("source") for m in existing["metadatas"] if m}

# Only embed chunks from sources not yet in the DB
new_splits = [doc for doc in all_splits if doc.metadata.get("source") not in indexed_sources]

if new_splits:
    vector_store.add_documents(new_splits)
    new_sources = {doc.metadata.get("source") for doc in new_splits}
    print(f"Indexed {len(new_splits)} new chunks from: {new_sources}")
else:
    print(f"All documents already indexed ({len(indexed_sources)} sources).")

All documents already indexed (1 sources).


In [6]:
ids = vector_store.add_documents(documents=all_splits)

In [7]:
results = vector_store.similarity_search(
    "How Rag is different from similarity search?"
)

print(results[0])

page_content='3. Direct Concept Comparison 
Feature Agent Tools Agent Memory 
Primary Purpose InteracƟng with the external world and 
data systems. 
Retaining context, conversaƟonal state, 
and preferences. 
InformaƟon Flow Pulls fresh external data in or pushes 
acƟons out. 
Retains internal conversaƟon history 
across Ɵme. 
How the LLM 
views it 
As a list of available acƟons with clear 
text descripƟons. 
As a backdrop of historical text or 
structured proﬁles. 
Examples Google Search, SQL Database Connector, 
Calculator. 
Chat Message History, Vector Database 
proﬁle store.' metadata={'page_label': '5', 'title': 'Microsoft Word - Concept of agent tools and memory in langchain', 'moddate': '2026-06-02T15:41:33+05:30', 'producer': 'Microsoft: Print To PDF', 'source': 'C:\\\\Users\\\\dell\\\\Documents\\\\Agentic-AI-Langchain\\\\Content\\Concept of agent tools and memory in langchain.pdf', 'page': 4, 'creationdate': '2026-06-02T15:41:33+05:30', 'author': '', 'start_index': 0, 'creator'

## RAG Agent

In [8]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [9]:

from tavily import TavilyClient
@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    tavily=TavilyClient()
    result = tavily.search(query)
    return result

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="llama3.2")

agent = create_agent(
    model=llm,
    tools=[search_handbook,search_web],
    system_prompt="You are a helpful agent that can search Agentic AI information from given documents if not found there then search the web.",
    checkpointer=InMemorySaver()
)

In [11]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="How langgraph state graph works with langsmith?")]}, 
    config=config
)

In [12]:
from pprint import pprint

pprint(response["messages"][-1].content)

('LangGraph and LangSmith are two tools in the LangChain ecosystem that work '
 'together to help developers build and deploy AI systems. LangGraph is a '
 'library for building and managing agents, while LangSmith is a tool for '
 'tracing and debugging these agents.\n'
 '\n'
 'To use LangSmith with LangGraph, you need to set up your environment '
 'variables correctly. This includes setting the LANGSMITH_TRACING variable to '
 'true, as well as setting the LANGSMITH_API_KEY and OPENAI_API_KEY variables '
 'to your API keys. You can also set the LANGSMITH_WORKSPACE_ID variable to '
 'specify which workspace to use.\n'
 '\n'
 'Once you have set up your environment variables, you can use LangSmith to '
 'trace and debug your agents. This involves creating a StateGraph object and '
 'adding nodes to it, one of which is the call model. The workflow then uses '
 'this state graph to visualize the tracing process.\n'
 '\n'
 'LangSmith provides comprehensive debugging capabilities, allowing 

In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="what is RAG?")]}, 
    config=config
)

In [14]:

pprint(response["messages"][-1].content)

('The concept of RAG (Reasoning Agent Graph) in LangChain refers to a visual '
 "representation of an agent's workflow, which includes its tools, memory, and "
 'reasoning process.\n'
 '\n'
 'In LangChain, agents are dynamic reasoning engines that use a Large Language '
 'Model (LLM) to decide on a sequence of actions. The RAG provides a graphical '
 'representation of this workflow, showing how the agent evaluates user input, '
 'decides which tools to use, executes them, and passes the results back into '
 'its memory to plan the next step.\n'
 '\n'
 'The RAG includes the following components:\n'
 '\n'
 '1. **Tools**: These are the "hands and feet" of the LLM, extending its '
 'capabilities by interacting with the outside world.\n'
 '2. **Memory**: This is where the agent stores information and context for '
 'the current task or conversation.\n'
 "3. **Reasoning Process**: This represents the agent's decision-making "
 'process, which involves evaluating user input, selecting tools,

In [15]:
response

{'messages': [HumanMessage(content='How langgraph state graph works with langsmith?', additional_kwargs={}, response_metadata={}, id='11cd0dcc-0c64-4514-9c01-1f5950985dfa'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-06-20T08:17:10.2442528Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5159556800, 'load_duration': 246329500, 'prompt_eval_count': 221, 'prompt_eval_duration': 2681792000, 'eval_count': 24, 'eval_duration': 2106226999, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019ee41a-d83a-7891-8992-cd1223a2b6d1-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langgraph state graph works with langsmith'}, 'id': 'a67216b9-5809-4bc0-9528-0e07726ec632', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 24, 'total_tokens': 245}),
  ToolMessage(content='{"query": "langgraph state graph works with langsmith", "follow_up_q